# Butterfly Classification with TensorFlow/Keras
Notebook ini melatih dua pendekatan: **Simple CNN** dan **Transfer Learning InceptionV3** dengan TensorFlow/Keras.

Struktur dibuat per tahap: setiap penjelasan di markdown, lalu code di bawahnya.
Output dibuat ringkas (minimal print), dan implementasi mengikuti best practice Keras dari referensi Context7 (tf.data pipeline, callback training, transfer learning).

## 1. Setup Environment dan Import Library

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input as inception_preprocess

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(42)
AUTOTUNE = tf.data.AUTOTUNE
device_name = tf.config.list_physical_devices('GPU')
'GPU available' if device_name else 'CPU only'

## 2. Konfigurasi Path Dataset dan Hyperparameter
Path dataset diset langsung ke direktori Kaggle agar lebih simple.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/kaggle/input/datasets/phucthaiv02/butterfly-image-classification')

TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
TRAIN_CSV = DATA_DIR / 'Training_set.csv'
TEST_CSV = DATA_DIR / 'Testing_set.csv'

BATCH_SIZE = 32
VAL_SIZE = 0.2

EPOCHS_CNN = 12
EPOCHS_INCEPTION_HEAD = 6
EPOCHS_INCEPTION_FINETUNE = 4

LR_CNN = 1e-3
LR_INCEPTION_HEAD = 1e-3
LR_INCEPTION_FINETUNE = 1e-5

IMG_SIZE_CNN = 224
IMG_SIZE_INCEPTION = 299

WORK_DIR = Path('/kaggle/working')
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

SIMPLE_CNN_CKPT = CHECKPOINT_DIR / 'best_simple_cnn.keras'
INCEPTION_CKPT = CHECKPOINT_DIR / 'best_inception_v3.keras'

PRED_CSV_PATH = WORK_DIR / 'test_predictions_tf.csv'
FINAL_MODEL_PATH = CHECKPOINT_DIR / 'best_final_model.keras'

TRAIN_DIR.exists(), TEST_DIR.exists(), TRAIN_CSV.exists(), TEST_CSV.exists()

## 3. Definisi Preprocessing dan Augmentasi Data
Train memakai augmentasi ringan. Untuk transfer learning InceptionV3, preprocessing memakai `keras.applications.inception_v3.preprocess_input`.

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomContrast(0.1),
], name="data_augmentation")

def decode_and_resize(path, label, img_size):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (img_size, img_size))
    img = tf.cast(img, tf.float32)
    return img, label

def preprocess_for_simplecnn(path, label, training=False):
    img, label = decode_and_resize(path, label, IMG_SIZE_CNN)
    if training:
        img = data_augmentation(img, training=True)
    img = img / 255.0
    return img, label

def preprocess_for_inception(path, label, training=False):
    img, label = decode_and_resize(path, label, IMG_SIZE_INCEPTION)
    if training:
        img = data_augmentation(img, training=True)
    img = inception_preprocess(img)
    return img, label

## 4. Load Dataset Train/Test dengan `tf.data`
Karena struktur gambar flat, label diambil dari CSV lalu dibangun jadi pipeline `tf.data` yang efisien (`cache`, `shuffle`, `prefetch`).

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

class_names = sorted(train_df['label'].unique().tolist())
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
num_classes = len(class_names)

train_df = train_df.copy()
train_df['label_idx'] = train_df['label'].map(class_to_idx)

try:
    tr_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=42,
        stratify=train_df['label_idx']
    )
except ValueError:
    tr_df, val_df = train_test_split(
        train_df,
        test_size=VAL_SIZE,
        random_state=42,
        stratify=None
    )

tr_df = tr_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

def make_paths_and_labels(df, image_dir, with_label=True):
    paths = (image_dir / df['filename']).astype(str).values
    if with_label:
        labels = df['label_idx'].astype('int32').values
        return paths, labels
    return paths

train_paths, train_labels = make_paths_and_labels(tr_df, TRAIN_DIR)
val_paths, val_labels = make_paths_and_labels(val_df, TRAIN_DIR)
test_paths = make_paths_and_labels(test_df, TEST_DIR, with_label=False)

def build_dataset(paths, labels=None, preprocess_fn=None, training=False):
    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices((paths, tf.zeros(len(paths), dtype=tf.int32)))
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if training:
        ds = ds.shuffle(min(len(paths), 2048), reshuffle_each_iteration=True)

    ds = ds.map(lambda x, y: preprocess_fn(x, y, training=training), num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds

## 5. Inspect Class Mapping dan Contoh Batch
Verifikasi mapping label, ukuran split, dan dimensi tensor sebelum training.

In [ ]:
train_ds_cnn = build_dataset(train_paths, train_labels, preprocess_for_simplecnn, training=True)
val_ds_cnn = build_dataset(val_paths, val_labels, preprocess_for_simplecnn, training=False)
test_ds_cnn = build_dataset(test_paths, None, preprocess_for_simplecnn, training=False)

print({'num_classes': num_classes, 'train_size': len(tr_df), 'val_size': len(val_df), 'test_size': len(test_df)})
print('sample class_to_idx:', dict(list(class_to_idx.items())[:8]))

images, labels = next(iter(train_ds_cnn))
fig, axes = plt.subplots(2, 4, figsize=(12, 6))

for i, ax in enumerate(axes.flatten()):
    img = tf.clip_by_value(images[i], 0.0, 1.0).numpy()
    ax.imshow(img)
    ax.set_title(idx_to_class[int(labels[i])][:18])
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6. Definisi Arsitektur Simple CNN (Keras)

In [ ]:
def build_simple_cnn(num_classes: int):
    model = keras.Sequential([
        layers.Input(shape=(IMG_SIZE_CNN, IMG_SIZE_CNN, 3)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation='softmax')
    ], name='simple_cnn')

    model.compile(
        optimizer=keras.optimizers.Adam(LR_CNN),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

simple_cnn = build_simple_cnn(num_classes)
simple_cnn.summary()

## 7. Training Simple CNN
Training menggunakan callback best practice: `ModelCheckpoint`, `EarlyStopping`, dan `ReduceLROnPlateau`.

In [ ]:
callbacks_cnn = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(SIMPLE_CNN_CKPT),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history_cnn = simple_cnn.fit(
    train_ds_cnn,
    validation_data=val_ds_cnn,
    epochs=EPOCHS_CNN,
    callbacks=callbacks_cnn,
    verbose=1
)

best_val_acc_cnn = float(np.max(history_cnn.history['val_accuracy']))
best_val_acc_cnn

## 8. Evaluasi Simple CNN (Validation Set)
`Testing_set.csv` tidak punya label, jadi evaluasi metrik klasifikasi dilakukan di validation set.

In [ ]:
val_probs_cnn = simple_cnn.predict(val_ds_cnn, verbose=0)
y_pred_cnn = val_probs_cnn.argmax(axis=1)
y_true_cnn = val_labels

acc_cnn = accuracy_score(y_true_cnn, y_pred_cnn)
print(f"SimpleCNN Validation Accuracy: {acc_cnn:.4f}")
print(classification_report(y_true_cnn, y_pred_cnn, target_names=class_names, zero_division=0))

cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_cnn, cmap='Blues')
plt.title('SimpleCNN Confusion Matrix (Validation)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 9. Definisi Transfer Learning InceptionV3 (Pretrained)
Tahap awal: freeze backbone InceptionV3, train classifier head dulu.

In [ ]:
train_ds_inc = build_dataset(train_paths, train_labels, preprocess_for_inception, training=True)
val_ds_inc = build_dataset(val_paths, val_labels, preprocess_for_inception, training=False)
test_ds_inc = build_dataset(test_paths, None, preprocess_for_inception, training=False)

base_model = InceptionV3(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE_INCEPTION, IMG_SIZE_INCEPTION, 3)
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE_INCEPTION, IMG_SIZE_INCEPTION, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

inception_model = keras.Model(inputs, outputs, name='inception_v3_transfer')
inception_model.compile(
    optimizer=keras.optimizers.Adam(LR_INCEPTION_HEAD),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

inception_model.summary()

## 10. Training InceptionV3 (Head Training + Fine-tuning Ringan)
Best practice transfer learning: train head dulu, lalu unfreeze sebagian backbone dengan learning rate kecil.

In [ ]:
callbacks_inc = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(INCEPTION_CKPT),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_inc_head = inception_model.fit(
    train_ds_inc,
    validation_data=val_ds_inc,
    epochs=EPOCHS_INCEPTION_HEAD,
    callbacks=callbacks_inc,
    verbose=1
)

base_model.trainable = True
for layer in base_model.layers[:-40]:
    layer.trainable = False

inception_model.compile(
    optimizer=keras.optimizers.Adam(LR_INCEPTION_FINETUNE),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_inc_ft = inception_model.fit(
    train_ds_inc,
    validation_data=val_ds_inc,
    epochs=EPOCHS_INCEPTION_FINETUNE,
    callbacks=callbacks_inc,
    verbose=1
)

best_val_acc_inc = max(
    float(np.max(history_inc_head.history['val_accuracy'])),
    float(np.max(history_inc_ft.history['val_accuracy']))
)
best_val_acc_inc

## 11. Evaluasi InceptionV3 dan Perbandingan Hasil Model

In [ ]:
inception_model = keras.models.load_model(INCEPTION_CKPT)

val_probs_inc = inception_model.predict(val_ds_inc, verbose=0)
y_pred_inc = val_probs_inc.argmax(axis=1)
y_true_inc = val_labels

acc_inc = accuracy_score(y_true_inc, y_pred_inc)
print(f"InceptionV3 Validation Accuracy: {acc_inc:.4f}")
print(classification_report(y_true_inc, y_pred_inc, target_names=class_names, zero_division=0))

cm_inc = confusion_matrix(y_true_inc, y_pred_inc)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_inc, cmap='Greens')
plt.title('InceptionV3 Confusion Matrix (Validation)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

compare_df = pd.DataFrame({
    'model': ['SimpleCNN', 'InceptionV3'],
    'val_accuracy': [acc_cnn, acc_inc],
    'best_val_during_training': [best_val_acc_cnn, best_val_acc_inc],
})
compare_df

## 12. Inferensi pada Data Test dan Simpan Prediksi/Model
Model terbaik dipilih dari validation accuracy, lalu dipakai untuk prediksi `Testing_set.csv`.

In [ ]:
if acc_inc >= acc_cnn:
    best_name = 'InceptionV3'
    best_model = keras.models.load_model(INCEPTION_CKPT)
    best_test_ds = test_ds_inc
else:
    best_name = 'SimpleCNN'
    best_model = keras.models.load_model(SIMPLE_CNN_CKPT)
    best_test_ds = test_ds_cnn

probs = best_model.predict(best_test_ds, verbose=0)
pred_idx = probs.argmax(axis=1)
conf = probs.max(axis=1)

pred_df = pd.DataFrame({
    'filename': test_df['filename'].values,
    'label': [idx_to_class[int(i)] for i in pred_idx],
    'confidence': conf,
})
pred_df.to_csv(PRED_CSV_PATH, index=False)

best_model.save(FINAL_MODEL_PATH)

print({'best_model': best_name, 'prediction_csv': str(PRED_CSV_PATH), 'final_model': str(FINAL_MODEL_PATH)})
pred_df.head()